# Lab 04 — Context Engineering and a Reusable Coding Skill

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-04-context-engineering-and-a-reusable-coding-skill/lab-04-context-engineering-and-a-reusable-coding-skill.ipynb)

**Topic:** 2 — Vibe Coding for Multi-Agent Systems

**Objective:** Engineer the agent's context and package a repeatable procedure as a project skill

Improve reliability by controlling exactly what the coding agent sees, then capture your house conventions as a skill so the agent applies them automatically.

Full step-by-step instructions are in the Learner Guide.


> **This lab has no Python to run.** The work is configuring a coding agent and comparing what it produces with and without context. This notebook holds the two artifacts — the context file and the skill file — so you can read and copy them.

In the repo folder they are `CLAUDE.md` and `skills/add-agent-tool.md`. You work inside the Lab 03 project, which already has Git history and a working agent.


## 0. Environment

This lab writes Markdown artifacts rather than running an agent, so it needs almost nothing installed. The install below is only for consistency with the other labs and for the Lab 03 project you will be editing.


**No API key cell in this lab.** Nothing here calls a model API — you are writing Markdown artifacts and directing a coding agent in your terminal. Labs that do call an API set their keys from Colab Secrets (`from google.colab import userdata`), never from a hard-coded string. The Lab 03 project you edit keeps its `OPENAI_API_KEY` in `.env`.


In [ ]:
!pip install -q python-dotenv


## 1. Baseline: a deliberately vague prompt with NO context file

Start a fresh session in the Lab 03 project with no context file present, and give the agent an underspecified request:

> Add a tool that fetches a stock price.

Record what is wrong. You will typically see several of these:

- The wrong file, or a brand-new file that duplicates existing structure.
- A different naming convention (`fetchStockPrice` vs `fetch_stock_price`).
- No type hints or docstring, so the tool schema is uninformative.
- A hard-coded API key or URL.
- No error handling on the network call.
- No registration in the existing tool list, so the tool is never callable.

Save the baseline, then discard the code so the comparison is fair:

```bash
git diff > /tmp/baseline-no-context.diff
git checkout . && git clean -fd
```


## 2. Create the project context file

The context file is read automatically at the start of every session — `CLAUDE.md` for Claude Code, `GEMINI.md` for Gemini CLI. Note the explicit prohibitions: they matter as much as the conventions.


In [ ]:
CONTEXT_FILE = """# Project context - Research Assistant Agent

## Stack
- Python 3.11+, standard library plus `openai` and `python-dotenv` only.
- No new dependencies without asking first.

## Layout
- `research_agent.py` - the agent loop, tool functions and tool schemas.
- `documents/` - the searchable `.txt` collection. Data only, never code.
- `SPEC.md` - the specification. The acceptance test in it is authoritative.

## Conventions
- `snake_case` for functions and variables; `UPPER_SNAKE` for module constants.
- Every tool function has full type hints and a one-line docstring - these
  become the schema the model sees, so they must describe *when* to call it.
- Every tool returns a JSON-serialisable `dict`. On failure it returns
  `{"error": "..."}` rather than raising.
- Standard library imports first, then third-party, then local. One blank
  line between groups.

## Never do this
- Never hard-code an API key, URL or secret. Read them via `os.getenv`.
- Never edit `SPEC.md` to make a failing test pass.
- Never add a network call other than the OpenAI API - the spec forbids it.
- Never reformat or refactor a file you were not asked to change.
"""

with open("CLAUDE.md", "w") as fh:
    fh.write(CONTEXT_FILE)
print(CONTEXT_FILE)


## 3. Re-run the identical prompt and compare

Start a **new** session so the context file is loaded fresh, and give the identical eight words: *Add a tool that fetches a stock price.*

Compare against `/tmp/baseline-no-context.diff`. With the context file in place the same prompt typically now produces the right file, `snake_case`, type hints and a docstring, `os.getenv` for configuration, an `{"error": ...}` path, registration in the existing tool list — and a question about the new dependency, because the context file said to ask.

**Nothing about the prompt changed. Only the context did.** That is the point of the lab.


## 4. Write a skill file capturing a repeatable procedure

A context file states *what is true about the project*. A skill states *how to perform a specific procedure*. Claude Code reads it from `.claude/skills/add-agent-tool/SKILL.md`; Gemini CLI users place the same content at `.gemini/skills/add-agent-tool/SKILL.md`.


In [ ]:
import os

SKILL = """---
name: add-agent-tool
description: >
  Use when adding a new tool/function to the research agent. Covers the
  function, its JSON schema, registration and the verification step.
---

# Add an agent tool

Follow these steps in order. Do not skip the verification step.

## Steps

1. Write the Python function in `research_agent.py`, below the existing
   tool functions. Full type hints, one-line docstring stating *when* the
   agent should call it.
2. Return a JSON-serialisable `dict`. Catch exceptions and return
   `{"error": "<short reason>"}` - never let a tool raise into the loop.
3. Read any configuration from `os.getenv`, with a sensible default. Never
   hard-code a value that differs between machines.
4. Add the JSON schema to the `TOOLS` list. The `description` must say when
   to call the tool, not merely what it does - this is the routing signal.
5. Register the function in `TOOL_REGISTRY` under exactly the schema `name`.
6. Verify: run the acceptance test in `SPEC.md`, then run one question that
   should trigger the new tool and confirm from the trace that it fired.

## Quality checks

- [ ] Type hints on every parameter and the return value.
- [ ] Docstring says *when* to call, not just what it does.
- [ ] Failure path returns `{"error": ...}` rather than raising.
- [ ] Schema `name`, `TOOL_REGISTRY` key and function name all match.
- [ ] Existing acceptance test still passes.
"""

os.makedirs(".claude/skills/add-agent-tool", exist_ok=True)
with open(".claude/skills/add-agent-tool/SKILL.md", "w") as fh:
    fh.write(SKILL)
print(SKILL)


## 5. Reference precise files and line ranges, not whole files

Context is a budget. Spending it on 400 lines the agent does not need makes it *less* reliable, not more — the relevant lines get diluted.

Wasteful:

> Here is my entire research_agent.py [400 lines pasted]. Add a currency conversion tool.

Precise:

> Add a currency conversion tool. Follow the same pattern as `search_documents` at `research_agent.py:34-58`, and register it in the `TOOLS` list at `research_agent.py:70-95`. Use the add-agent-tool skill.

Find the line numbers quickly with:

```bash
grep -n "def search_documents\|^TOOLS\|^TOOL_REGISTRY" research_agent.py
```


## 6. Invoke the skill and check it against its own checklist

> Use the add-agent-tool skill to add `word_count(filename)`, which returns the number of words in a document under `documents/`.

Then verify the generated code satisfies every box in the skill's quality-checks list, and confirm the agent actually ran step 6 — it should have executed the acceptance test rather than merely claiming the tool works.

```bash
git diff
python research_agent.py "How many words are in mrt.txt?"
python research_agent.py "When did the Thomson-East Coast Line open?"
```

The second command re-runs the original acceptance test, confirming the new tool did not regress the existing behaviour.


## What you learned

- Context engineering moves reliability more than prompt wording.
- A context file (`CLAUDE.md` / `GEMINI.md`) encodes what is permanently true about the project, including explicit prohibitions.
- A skill encodes a repeatable procedure with its own quality checks.
- Precise file and line references beat pasting whole files: context is a budget.
- Baseline-then-compare is how you demonstrate a context improvement rather than assume it.
